# Notebook 6 - Transformer enrichi

Ce notebook est une experience d'amelioration apres le projet initial. Il reprend l'idee du notebook 4, mais corrige sa principale limite : le Transformer n'etait entraine que sur un tres petit echantillon.

Objectif : fine-tuner un Transformer sur la taxonomie enrichie du notebook 5, avec davantage de donnees et une evaluation comparable sur le domaine NovelForge.

## Positionnement

Les notebooks 1 a 4 constituent le projet initial : EDA, baseline ML, LSTM et Transformer de demonstration.

Les notebooks 5 et 6 sont des experiences d'amelioration :

- notebook 5 : meilleur travail sur les donnees, taxonomie regroupee, baseline enrichie ;
- notebook 6 : tentative de Transformer plus serieux avec le dataset enrichi.

Le Transformer est le modele le plus puissant en theorie, mais il a besoin de plus de donnees et de fine-tuning pour exprimer cet avantage.

In [1]:
from pathlib import Path
import os
import sys
import time

# Works whether Jupyter starts from the project root or from notebooks/.
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent
elif not (PROJECT_DIR / 'src').exists() and (PROJECT_DIR.parent / 'src').exists():
    PROJECT_DIR = PROJECT_DIR.parent

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import joblib
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import classification_report, f1_score, hamming_loss, jaccard_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

from src.baseline_ml import apply_thresholds, find_best_global_threshold, find_best_label_thresholds
from src.enriched_dataset import build_enriched_dataset, load_original_dataset, summarize_labels
from src.project_config import ENRICHED_GENRE_VOCABULARY
from src.transformer_model import NovelForgeTransformer, TransformerConfig

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 140)

DEVICE_KIND = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device available: {DEVICE_KIND}')
PROJECT_DIR

Device available: cpu


WindowsPath('C:/Users/ClémentPERRET/OneDrive - EQUATERRE-VDS/Bureau/Cours/DeepLearning')

## Parametres ajustables

Les valeurs ci-dessous sont un compromis. Sur CPU, elles restent plus lourdes que le notebook 4 mais encore raisonnables. Sur GPU, on peut augmenter les lignes et le nombre d'epochs.

In [2]:
if DEVICE_KIND == 'cuda':
    CURRENT_TRAIN_ROWS = 8_000
    MAL_TRAIN_ROWS = 12_000
    VALID_ROWS = 2_000
    TEST_ROWS = 3_000
    EPOCHS = 3
    TRAIN_BATCH_SIZE = 16
    EVAL_BATCH_SIZE = 32
else:
    CURRENT_TRAIN_ROWS = 2_000
    MAL_TRAIN_ROWS = 3_000
    VALID_ROWS = 800
    TEST_ROWS = 1_200
    EPOCHS = 2
    TRAIN_BATCH_SIZE = 8
    EVAL_BATCH_SIZE = 16

MAX_LENGTH = 192
MODEL_NAME = 'distilbert-base-uncased'
RANDOM_STATE = 42

print({
    'current_train_rows': CURRENT_TRAIN_ROWS,
    'mal_train_rows': MAL_TRAIN_ROWS,
    'valid_rows': VALID_ROWS,
    'test_rows': TEST_ROWS,
    'epochs': EPOCHS,
    'batch_size': TRAIN_BATCH_SIZE,
    'max_length': MAX_LENGTH,
})

{'current_train_rows': 2000, 'mal_train_rows': 3000, 'valid_rows': 800, 'test_rows': 1200, 'epochs': 2, 'batch_size': 8, 'max_length': 192}


## Chargement des donnees enrichies

On garde NovelForge comme domaine d'evaluation. Le train est enrichi avec un echantillon de `manga_dataset.csv`, plus proche du domaine manga/manhwa que l'anime.

In [3]:
current = load_original_dataset(PROJECT_DIR / 'data' / 'data.csv')
mal_manga = build_enriched_dataset(PROJECT_DIR, include_original=False, include_manga=True, include_anime=False)

print(f'Dataset courant exploitable : {len(current):,}')
print(f'MAL manga exploitable : {len(mal_manga):,}')

display(summarize_labels(current).head(12))
display(summarize_labels(mal_manga).head(12))

Dataset courant exploitable : 69,588
MAL manga exploitable : 56,587


,label,count
0,Romance,29762
1,Comedy,21282
2,Drama,18702
3,Fantasy,16125
4,BL/GL Romance,14659
5,Action,12734
6,School Life,12582
7,Seinen,9235
8,Supernatural,8689
9,Shoujo,8340


,label,count
0,Adult,19214
1,Romance,14881
2,BL/GL Romance,12045
3,Comedy,11665
4,Fantasy,10987
5,Drama,9478
6,School Life,9073
7,Action,8045
8,Shoujo,7211
9,Supernatural,7035


In [4]:
current_train_full, current_temp = train_test_split(current, test_size=0.30, random_state=RANDOM_STATE, shuffle=True)
current_valid_full, current_test_full = train_test_split(current_temp, test_size=0.50, random_state=RANDOM_STATE, shuffle=True)

current_train = current_train_full.sample(n=min(CURRENT_TRAIN_ROWS, len(current_train_full)), random_state=RANDOM_STATE)
mal_train = mal_manga.sample(n=min(MAL_TRAIN_ROWS, len(mal_manga)), random_state=RANDOM_STATE)
valid_df = current_valid_full.sample(n=min(VALID_ROWS, len(current_valid_full)), random_state=RANDOM_STATE)
test_df = current_test_full.sample(n=min(TEST_ROWS, len(current_test_full)), random_state=RANDOM_STATE)

train_df = pd.concat([current_train, mal_train], ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'Train courant sample : {len(current_train):,}')
print(f'Train MAL sample : {len(mal_train):,}')
print(f'Train total Transformer : {len(train_df):,}')
print(f'Validation NovelForge : {len(valid_df):,}')
print(f'Test NovelForge : {len(test_df):,}')

Train courant sample : 2,000
Train MAL sample : 3,000
Train total Transformer : 5,000
Validation NovelForge : 800
Test NovelForge : 1,200


In [5]:
mlb = MultiLabelBinarizer(classes=ENRICHED_GENRE_VOCABULARY)
mlb.fit([ENRICHED_GENRE_VOCABULARY])
labels = list(mlb.classes_)
id2label = {index: label for index, label in enumerate(labels)}
label2id = {label: index for index, label in id2label.items()}

X_train = train_df['synopsis_clean'].fillna('').astype(str).to_numpy()
X_valid = valid_df['synopsis_clean'].fillna('').astype(str).to_numpy()
X_test = test_df['synopsis_clean'].fillna('').astype(str).to_numpy()

y_train = mlb.transform(train_df['genre_labels']).astype('float32')
y_valid = mlb.transform(valid_df['genre_labels']).astype('float32')
y_test = mlb.transform(test_df['genre_labels']).astype('float32')

print(f'Labels enrichis : {len(labels)}')
labels

Labels enrichis : 26


['Action',
 'Adventure',
 'Comedy',
 'Drama',
 'Fantasy',
 'Romance',
 'BL/GL Romance',
 'Adult',
 'School Life',
 'Slice of Life',
 'Supernatural',
 'Mystery',
 'Psychological',
 'Horror',
 'Historical',
 'Sci Fi',
 'Sports',
 'Martial Arts',
 'Magic',
 'Isekai',
 'Harem',
 'Mecha',
 'Seinen',
 'Shoujo',
 'Shounen',
 'Josei']

## Fine-tuning du Transformer

Le modele sauvegarde ses checkpoints dans `models/transformer_enriched_novelforge`. Cette experience ne remplace pas le Transformer minimal du notebook 4.

In [6]:
output_dir = PROJECT_DIR / 'models' / 'transformer_enriched_novelforge'
config = TransformerConfig(
    model_name=MODEL_NAME,
    max_length=MAX_LENGTH,
    learning_rate=2e-5,
    train_batch_size=TRAIN_BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE,
    epochs=EPOCHS,
    weight_decay=0.01,
    threshold=0.5,
    output_dir=str(output_dir),
    logging_steps=50,
)

transformer = NovelForgeTransformer(
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    config=config,
)

start = time.perf_counter()
trainer = transformer.fine_tune(X_train, y_train, X_valid, y_valid)
training_seconds = time.perf_counter() - start
print(f'Training time: {training_seconds:.1f} seconds')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\ClémentPERRET\OneDrive - EQUATERRE-VDS\Bureau\Cours\DeepLearning\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then de

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.290235,0.292454,0.177318,0.048420
2,0.267579,0.281685,0.229827,0.083548


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\ClémentPERRET\OneDrive - EQUATERRE-VDS\Bureau\Cours\DeepLearning\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training time: 2423.2 seconds


## Calibration des seuils

Comme pour la baseline enrichie, on ne se contente pas du seuil `0.5`. On cherche un seuil global puis des seuils par label sur la validation.

In [7]:
valid_proba = transformer.predict_proba(X_valid, batch_size=EVAL_BATCH_SIZE)
test_proba = transformer.predict_proba(X_test, batch_size=EVAL_BATCH_SIZE)

best_global_threshold, valid_f1_micro = find_best_global_threshold(y_valid, valid_proba)
label_thresholds = find_best_label_thresholds(y_valid, valid_proba)

print(f'Best global threshold: {best_global_threshold:.2f}')
print(f'Validation F1 micro at best global threshold: {valid_f1_micro:.4f}')
print('First label thresholds:', dict(zip(labels[:8], label_thresholds[:8])))

c:\Users\ClémentPERRET\OneDrive - EQUATERRE-VDS\Bureau\Cours\DeepLearning\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Best global threshold: 0.20
Validation F1 micro at best global threshold: 0.4264
First label thresholds: {'Action': 0.25, 'Adventure': 0.2, 'Comedy': 0.2, 'Drama': 0.2, 'Fantasy': 0.35, 'Romance': 0.3, 'BL/GL Romance': 0.2, 'Adult': 0.15}


In [8]:
def evaluate_probabilities(y_true, probabilities, thresholds):
    y_pred = apply_thresholds(probabilities, thresholds)
    return {
        'f1_micro': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'jaccard_samples': jaccard_score(y_true, y_pred, average='samples', zero_division=0),
        'hamming_loss': hamming_loss(y_true, y_pred),
        'classification_report_text': classification_report(y_true, y_pred, target_names=labels, zero_division=0),
        'classification_report': classification_report(y_true, y_pred, target_names=labels, zero_division=0, output_dict=True),
    }

metrics_global = evaluate_probabilities(y_test, test_proba, best_global_threshold)
metrics_per_label = evaluate_probabilities(y_test, test_proba, label_thresholds)

summary = pd.DataFrame([
    {
        'model': 'transformer_enriched',
        'threshold_strategy': 'global',
        'train_rows': len(train_df),
        'current_train_rows': len(current_train),
        'mal_train_rows': len(mal_train),
        'valid_rows': len(valid_df),
        'test_rows': len(test_df),
        'threshold': best_global_threshold,
        'valid_f1_micro': valid_f1_micro,
        'training_seconds': training_seconds,
        **{key: metrics_global[key] for key in ['f1_micro', 'f1_macro', 'f1_weighted', 'jaccard_samples', 'hamming_loss']},
    },
    {
        'model': 'transformer_enriched',
        'threshold_strategy': 'per_label',
        'train_rows': len(train_df),
        'current_train_rows': len(current_train),
        'mal_train_rows': len(mal_train),
        'valid_rows': len(valid_df),
        'test_rows': len(test_df),
        'threshold': None,
        'valid_f1_micro': None,
        'training_seconds': training_seconds,
        **{key: metrics_per_label[key] for key in ['f1_micro', 'f1_macro', 'f1_weighted', 'jaccard_samples', 'hamming_loss']},
    },
])
summary

,model,threshold_strategy,train_rows,current_train_rows,mal_train_rows,valid_rows,test_rows,threshold,valid_f1_micro,training_seconds,f1_micro,f1_macro,f1_weighted,jaccard_samples,hamming_loss
0,transformer_enriched,global,5000,2000,3000,800,1200,0.2,0.426421,2423.217297,0.429954,0.201669,0.373770,0.280034,0.164199
1,transformer_enriched,per_label,5000,2000,3000,800,1200,NaN,NaN,2423.217297,0.400670,0.269212,0.421666,0.239096,0.194936


In [9]:
report = pd.DataFrame(metrics_per_label['classification_report']).T
report.loc[labels, ['precision', 'recall', 'f1-score', 'support']].sort_values('f1-score', ascending=False)

,precision,recall,f1-score,support
Romance,0.632353,0.670565,0.650899,513.0
BL/GL Romance,0.586716,0.606870,0.596623,262.0
Fantasy,0.575107,0.525490,0.549180,255.0
Action,0.500000,0.587379,0.540179,206.0
School Life,0.500000,0.544186,0.521158,215.0
Comedy,0.308113,0.952113,0.465565,355.0
Drama,0.345779,0.659443,0.453674,323.0
Adventure,0.335484,0.504854,0.403101,103.0
Josei,0.377551,0.389474,0.383420,95.0
Shoujo,0.236948,0.440299,0.308094,134.0


## Sauvegarde

Ces artefacts sont separes de ceux du notebook 4. Streamlit detecte automatiquement ce Transformer enrichi s'il existe.

In [10]:
models_dir = PROJECT_DIR / 'models'
reports_dir = PROJECT_DIR / 'reports'
models_dir.mkdir(exist_ok=True)
reports_dir.mkdir(exist_ok=True)

transformer.save(output_dir)
joblib.dump(labels, models_dir / 'transformer_enriched_labels.joblib')
joblib.dump(label_thresholds, models_dir / 'transformer_enriched_thresholds.joblib')
joblib.dump(
    {
        'summary': summary,
        'labels': labels,
        'global_threshold': best_global_threshold,
        'label_thresholds': label_thresholds,
        'metrics_global': metrics_global,
        'metrics_per_label': metrics_per_label,
        'config': config.__dict__,
    },
    models_dir / 'transformer_enriched_metrics.joblib',
)
summary.to_csv(reports_dir / 'transformer_enriched_metrics.csv', index=False)

print(f'Transformer enrichi sauvegarde : {output_dir}')
print('Labels/seuils/metriques sauvegardes dans models/')
print('Resume sauvegarde dans reports/transformer_enriched_metrics.csv')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Transformer enrichi sauvegarde : C:\Users\ClémentPERRET\OneDrive - EQUATERRE-VDS\Bureau\Cours\DeepLearning\models\transformer_enriched_novelforge
Labels/seuils/metriques sauvegardes dans models/
Resume sauvegarde dans reports/transformer_enriched_metrics.csv


## Conclusion

Ce notebook teste l'hypothese du cours dans de meilleures conditions que le notebook 4 : plus de donnees, taxonomie regroupee et calibration des seuils.

Si les scores depassent la baseline enrichie du notebook 5, le Transformer devient le meilleur modele empirique du projet. Sinon, la conclusion reste interessante : le Transformer est plus puissant en theorie, mais son avantage depend fortement du volume de donnees, du temps d'entrainement et des ressources de calcul disponibles.

Dans les deux cas, cette experience separe clairement le projet initial des ameliorations exploratoires.